In [1]:
import pandas as pd
import numpy as np
import random
import copy
import math
from datetime import date
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

# KINGElvis FileName
file_name="VTA_CA_OB_KINGElvis.xlsx"

if file_name.split('_')[0].isdigit():
    file_first_name=file_name.split('_')[0]+'_'+file_name.split('_')[1]
else:
    file_first_name=file_name.split('_')[0]

# in some Compeletion Report LSNAMECODE is splited in some it is not so have to check that
def edit_ls_code_column(x):
    value=x.split('_')
    if len(value)>3:
        route_value="_".join(value[:-1])
    else:
        route_value="_".join(value)
    return route_value

# for generated file version
version=2
project_name='TUCSON'
today_date = date.today()
today_date=''.join(str(today_date).split('-'))


In [2]:
detail_df_stops=pd.read_excel('details_TUCSON_AZ_od_excel.xlsx',sheet_name='STOPS')
detail_df_xfers=pd.read_excel('details_TUCSON_AZ_od_excel.xlsx',sheet_name='XFERS')

wkend_overall_df=pd.read_excel('TUCSON_AZ_CR.xlsx',sheet_name='WkEND-Overall')
# wkend_overall_df['LS_NAME_CODE']=wkend_overall_df['LS_NAME_CODE'].apply(edit_ls_code_column)
wkend_route_df=pd.read_excel('TUCSON_AZ_CR.xlsx',sheet_name='WkEND-RouteTotal')
df=pd.read_csv("reviewtool_20250228_TUCSON_ROUTE_DIRECTION_CHECk.csv")
ke_df=pd.read_excel("Tucson_az_2025_KINGElvis (1).xlsx",sheet_name='Elvis_Review')

# if we have generated route_direction_database file using route_direction_refator_database.py file then have to replace and rename the columns
df.drop(columns=['ROUTE_SURVEYEDCode','ROUTE_SURVEYED'],inplace=True)
df.rename(columns={'ROUTE_SURVEYEDCode_New':'ROUTE_SURVEYEDCode','ROUTE_SURVEYED_NEW':'ROUTE_SURVEYED'},inplace=True) 

wkday_overall_df=pd.read_excel('TUCSON_AZ_CR.xlsx',sheet_name='WkDAY-Overall')
# wkday_overall_df['LS_NAME_CODE']=wkday_overall_df['LS_NAME_CODE'].apply(edit_ls_code_column)
wkday_route_df=pd.read_excel('TUCSON_AZ_CR.xlsx',sheet_name='WkDAY-RouteTotal')
# df.rename(columns={'ROUTE_SURVEYEDCode_x':'ROUTE_SURVEYEDCode','ROUTE_SURVEYED_x':'ROUTE_SURVEYED'},inplace=True) 

df['ROUTE_SURVEYEDCode_Splited']=df['ROUTE_SURVEYEDCode'].apply(lambda x:('_').join(str(x).split('_')[:-1]) )

In [3]:
df['ROUTE_SURVEYEDCode_Splited']=df['ROUTE_SURVEYEDCode'].apply(lambda x:('_').join(str(x).split('_')[:-1]) )


ke_df=ke_df[ke_df['INTERV_INIT']!='999']
ke_df=ke_df[ke_df['INTERV_INIT']!=999]
ke_df=ke_df[ke_df['1st Cleaner']!='No 5 MIN']
ke_df=ke_df[ke_df['1st Cleaner']!='Test']
ke_df=ke_df[ke_df['1st Cleaner']!='Test/No 5 MIN']
ke_df=ke_df[ke_df['Final_Usage'].str.lower()=='use']



# Getting Data from Database where the Final Usage is Use in KINGELVIS  
df=pd.merge(df,ke_df['id'],on='id',how='inner')
df=df[df['INTERV_INIT']!='999']
df=df[df['INTERV_INIT']!=999]
df.drop_duplicates(subset='id',inplace=True)

In [4]:
def check_all_characters_present(df, columns_to_check):
    # Function to clean a string by removing underscores and square brackets and converting to lowercase
    def clean_string(s):
        return s.replace('_', '').replace('[', '').replace(']', '').replace(' ','').replace('#','').lower()

    # Clean and convert all column names in df to lowercase for case-insensitive comparison
    df_columns_lower = [clean_string(column) for column in df.columns]

    # Clean and convert the columns_to_check list to lowercase for case-insensitive comparison
    columns_to_check_lower = [clean_string(column) for column in columns_to_check]

    # Use a list comprehension to filter columns
    matching_columns = [column for column in df.columns if clean_string(column) in columns_to_check_lower]

    return matching_columns

date_columns_check=['completed','datestarted']
date_columns=check_all_characters_present(df,date_columns_check)

def determine_date(row):
    if not pd.isnull(row[date_columns[0]]):
        return row[date_columns[0]]
    elif not pd.isnull(row[date_columns[1]]):
        return row[date_columns[1]]
    else:
        return pd.NaT

df['Date'] = df.apply(determine_date, axis=1)

In [5]:
def get_day_name(x):
    formats_to_check = ['%Y-%m-%d %H:%M:%S', '%d/%m/%Y %H:%M']
    
    for format_str in formats_to_check:
        try:
            date_object = datetime.strptime(x, format_str)
            day_name = date_object.strftime('%A')
            return day_name
        except ValueError:
            continue

df['Day']=df['Date'].apply(get_day_name)


# df['LAST_SURVEY_DATE'] = pd.to_datetime(df['Date'], format='%d/%m/%Y %H:%M')
# latest_date = df['LAST_SURVEY_DATE'].max()
# latest_date_df = pd.DataFrame({'Latest_Survey_Date': [latest_date]})

df['LAST_SURVEY_DATE'] = pd.to_datetime(df['Date'], format='%Y-%m-%d %H:%M:%S')
latest_date = df['LAST_SURVEY_DATE'].max()
latest_date_df = pd.DataFrame({'Latest_Survey_Date': [latest_date]})
df.to_csv('WEEK Wise File.csv',index=False)

weekend_df=df[df['Day'].isin(['Saturday','Sunday'])]
weekend_df.to_csv('WeekEnd_Data.csv',index=False)
weekday_df=df[~(df['Day'].isin(['Saturday','Sunday']))]

In [6]:
weekend_df[['Day']]

,Day
101,Saturday
102,Saturday
103,Saturday
104,Saturday
105,Saturday
...,...
5047,Sunday
5048,Sunday
5049,Sunday
5050,Sunday


In [7]:
time_column_check=['timeoncode']
time_period_column_check=['timeon']
time_column=check_all_characters_present(df,time_column_check)
time_period_column=check_all_characters_present(df,time_period_column_check)
route_survey_column_check=['routesurveyedcode']
route_survey_column=check_all_characters_present(df,route_survey_column_check)
stopon_clntid_column_check=['stoponclntid']
stopon_clntid_column=check_all_characters_present(df,stopon_clntid_column_check)
stopoff_clntid_column_check=['stopoffclntid']
stopoff_clntid_column=check_all_characters_present(df,stopoff_clntid_column_check)


In [8]:
weekend_df=df[df['Day'].isin(['Saturday','Sunday'])]
weekend_df.to_csv('WeekEnd_Data.csv',index=False)
weekday_df=df[~(df['Day'].isin(['Saturday','Sunday']))]
# weekday_df.to_csv('WeekEnd_Data.csv',index=False)

In [9]:
def create_time_value_df_with_display(df):
    """
    Create a time-value DataFrame summarizing counts and time ranges.

    Parameters:
        df (pd.DataFrame): Input DataFrame containing the time values.
        time_column (str): Name of the column in the input DataFrame containing the time values.

    Returns:
        pd.DataFrame: Processed DataFrame with counts, time ranges, and display text.
    """
    # Define time value groups
    # pre_early_am_values = ['AM1']
    # early_am_values = ['AM2']
    # am_values = ['AM3', 'AM4', 'MID1', 'MID2', 'MID7']
    # midday_values = ['MID3', 'MID4', 'MID5', 'MID6', 'PM1']
    # pm_values = ['PM2', 'PM3', 'PM4', 'PM5']
    # evening_values = ['PM6', 'PM7', 'PM8', 'PM9']

    # For TUCSIN PROJECT HAVE TO CHANGNE TIME-PERIODS values
    # pre_early_am_values = ['AM1']
    # early_am_values = ['AM2']
    am_values = ['AM1','AM2','AM3', 'AM4']
    midday_values = [ 'MID1', 'MID2','MID3', 'MID4', 'MID5', 'MID6' ]
    pm_values = ['PM1','PM2', 'PM3']
    evening_values = ['OFF1', 'OFF2', 'OFF3', 'OFF4','OFF5']

    # Mapping time groups to corresponding columns
    # time_group_mapping = {
    #     0: pre_early_am_values,
    #     1: early_am_values,
    #     2: am_values,
    #     3: midday_values,
    #     4: pm_values,
    #     5: evening_values,
    # }
    # for TUCSON have to change time groups too
    time_group_mapping = {
        1: am_values,
        2: midday_values,
        3: pm_values,
        4: evening_values,
    }

    # Mapping time values to time ranges
    # time_mapping = {
    #     'AM1': 'Before 5:00 am',
    #     'AM2': '5:00 am - 6:00 am',
    #     'AM3': '6:00 am - 7:00 am',
    #     'MID1': '7:00 am - 8:00 am',
    #     'MID2': '8:00 am - 9:00 am',
    #     'MID7': '9:00 am - 10:00 am',
    #     'MID3': '10:00 am - 11:00 am',
    #     'MID4': '11:00 am - 12:00 pm',
    #     'MID5': '12:00 pm - 1:00 pm',
    #     'MID6': '1:00 pm - 2:00 pm',
    #     'PM1': '2:00 pm - 3:00 pm',
    #     'PM2': '3:00 pm - 4:00 pm',
    #     'PM3': '4:00 pm - 5:00 pm',
    #     'PM4': '5:00 pm - 6:00 pm',
    #     'PM5': '6:00 pm - 7:00 pm',
    #     'PM6': '7:00 pm - 8:00 pm',
    #     'PM7': '8:00 pm - 9:00 pm',
    #     'PM8': '9:00 pm - 10:00 pm',
    #     'PM9': 'After 10:00 pm'
    # }

    time_mapping = {
        'AM1': 'Before 5:30 am',
        'AM2': '5:30 am - 6:30 am',
        'AM3': '6:30 am - 7:30 am',
        'AM4': '7:30 am - 8:30 am',
        'MID1': '8:30 am - 9:30 am',
        'MID2': '9:30 am - 10:30 am',
        'MID3': '10:30 am - 11:30 am',
        'MID4': '11:30 am - 12:30 pm',
        'MID5': '12:30 pm - 1:30 pm',
        'MID6': '1:30 pm - 2:30 pm',
        'PM1': '2:30 pm - 3:30 pm',
        'PM2': '3:30 pm - 4:30 pm',
        'PM3': '4:30 pm - 5:30 pm',
        'OFF1': '5:30 pm - 6:30 pm',
        'OFF2': '6:30 pm - 7:30 pm',
        'OFF3': '7:30 pm - 8:30 pm',
        'OFF4': '8:30 pm - 9:30 pm',
        'OFF5': 'After 9:30 pm'
    }

    # Initialize the new DataFrame
    new_df = pd.DataFrame(columns=["Original Text",  1, 2, 3, 4])

    # Populate the DataFrame with counts
    for col, values in time_group_mapping.items():
        for value in values:
            count = df[df[time_column[0]] == value].shape[0]
            row = {"Original Text": value}

            # Initialize all columns to 0
            for c in range(6):
                row[c] = 0

            # Update the corresponding column with the count
            row[col] = count
            new_df = pd.concat([new_df, pd.DataFrame([row])], ignore_index=True)

    # Map time values to time ranges
    new_df['Time Range'] = new_df['Original Text'].map(time_mapping)

    # Drop rows with missing time ranges
    new_df.dropna(subset=['Time Range'], inplace=True)

    # Add a display text column with sequential numbering
    new_df['Display_Text'] = range(1, len(new_df) + 1)

    return new_df

wkend_time_value_df=create_time_value_df_with_display(weekend_df)
wkday_time_value_df=create_time_value_df_with_display(weekday_df)


In [10]:
wkend_overall_df.dropna(subset=['LS_NAME_CODE'],inplace=True)
wkday_overall_df.dropna(subset=['LS_NAME_CODE'],inplace=True)


In [20]:

# To create Route_SurveyedCode Direction wise comparison in terms of time values
def create_route_direction_level_df(overalldf,df):
    #For Project other than TUCSON HAVE to change/uncomment the code 

    # pre_early_am_values=['AM1'] 
    # early_am_values=['AM2'] 
    # am_values=['AM3','AM4','MID1','MID2','MID7'] 
    # midday_values=['MID3','MID4','MID5','MID6','PM1']
    # pm_values=['PM2','PM3','PM4','PM5']
    # evening_values=['PM6','PM7','PM8','PM9']
    # pre_early_am_column=[0]  #0 is for Pre-Early AM header
    # early_am_column=[1]  #1 is for Early AM header
    # am_column=[2] #This is for AM header
    # midday_colum=[3] #this is for MIDDAY header
    # pm_column=[4] #this is for PM header
    # evening_column=[5] #this is for EVENING header

    # For Tucson PROJECT Have to change values TIME PERIOD VALUES

    am_values = ['AM1','AM2','AM3', 'AM4']
    midday_values = [ 'MID1', 'MID2','MID3', 'MID4', 'MID5', 'MID6' ]
    pm_values = ['PM1','PM2', 'PM3']
    evening_values = ['OFF1', 'OFF2', 'OFF3', 'OFF4','OFF5']
    am_column=[1] #This is for AM header
    midday_colum=[2] #this is for MIDDAY header
    pm_column=[3] #this is for PM header
    evening_column=[4] #this is for EVENING header

    def convert_string_to_integer(x):
        try:
            return float(x)
        except (ValueError, TypeError):
            return 0
        
    # Creating new dataframe for specifically AM, PM, MIDDAY, Evenving data and added values from Compeletion Report

    # For TUCSON PROJECT we are not using pre_early_am and early_am columns so have to comment the following code accordingly 
    new_df=pd.DataFrame()
    new_df['ROUTE_SURVEYEDCode']=overalldf['LS_NAME_CODE']
    new_df['Day'] = overalldf['DAY']

    # new_df['CR_PRE_Early_AM']=overalldf[pre_early_am_column[0]].apply(math.ceil)
    # new_df['CR_Early_AM']=overalldf[early_am_column[0]].apply(math.ceil)
    new_df['CR_AM_Peak']=overalldf[am_column[0]].apply(math.ceil)
    new_df['CR_Midday']=overalldf[midday_colum[0]].apply(math.ceil)
    new_df['CR_PM_Peak']=overalldf[pm_column[0]].apply(math.ceil)
    new_df['CR_Evening']=overalldf[evening_column[0]].apply(math.ceil)
    # print("new_df_columns",new_df.columns)
    # new_df[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']]=new_df[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']].applymap(convert_string_to_integer)
    new_df[['CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']]=new_df[['CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']].applymap(convert_string_to_integer)
    new_df.fillna(0,inplace=True)

    for index, row in new_df.iterrows():
        route_code = row['ROUTE_SURVEYEDCode']
        day=row['Day']
        def get_counts_and_ids(time_values):
            # Just for SALEM
            # subset_df = df[(df['ROUTE_SURVEYEDCode_Splited'] == route_code) & (df[time_column[0]].isin(time_values))]
            subset_df = df[(df['ROUTE_SURVEYEDCode'] == route_code) & (df[time_column[0]].isin(time_values))& 
                         (weekend_df['Day'].str.lower() == str(day).lower())]
            subset_df=subset_df.drop_duplicates(subset='id')
            count = subset_df.shape[0]
            ids = subset_df['id'].values
            return count, ids

        # pre_early_am_value, pre_early_am_value_ids = get_counts_and_ids(pre_early_am_values)
        # early_am_value, early_am_value_ids = get_counts_and_ids(early_am_values)
        am_value, am_value_ids = get_counts_and_ids(am_values)
        midday_value, midday_value_ids = get_counts_and_ids(midday_values)
        pm_value, pm_value_ids = get_counts_and_ids(pm_values)
        evening_value, evening_value_ids = get_counts_and_ids(evening_values)
        
        # new_df.loc[index, 'CR_Total'] = row['CR_PRE_Early_AM']+row['CR_Early_AM']+row['CR_AM_Peak'] + row['CR_Midday'] + row['CR_PM_Peak'] + row['CR_Evening']
        new_df.loc[index, 'CR_Total'] = row['CR_AM_Peak'] + row['CR_Midday'] + row['CR_PM_Peak'] + row['CR_Evening']
        new_df.loc[index, 'CR_AM_Peak'] =row['CR_AM_Peak']
        # new_df.loc[index, 'CR_AM_Peak'] =row['CR_PRE_EARLY_AM']+row['CR_EARLY_AM']+ row['CR_AM_Peak']

        # new_df.loc[index, 'DB_PRE_Early_AM_Peak'] = pre_early_am_value
        # new_df.loc[index, 'DB_Early_AM_Peak'] = early_am_value
        new_df.loc[index, 'DB_AM_Peak'] = am_value
        new_df.loc[index, 'DB_Midday'] = midday_value
        new_df.loc[index, 'DB_PM_Peak'] = pm_value
        new_df.loc[index, 'DB_Evening'] = evening_value
        # new_df.loc[index, 'DB_Total'] = evening_value + am_value + midday_value + pm_value+pre_early_am_value+early_am_value
        new_df.loc[index, 'DB_Total'] = evening_value + am_value + midday_value + pm_value
        
    #     new_df['ROUTE_SURVEYEDCode_Splited']=new_df['ROUTE_SURVEYEDCode'].apply(lambda x:('_').join(x.split('_')[:-1]) )

        # weekend_df.rename(columns={'ROUTE_TOTAL':'CR_Overall_Goal','SURVEY_ROUTE_CODE':'ROUTE_SURVEYEDCode','LS_NAME_CODE':'ROUTE_SURVEYEDCode'},inplace=True)

        for index, row in new_df.iterrows():
            # pre_early_am_peak_diff=row['CR_PRE_Early_AM']-row['DB_PRE_Early_AM_Peak']
            # early_am_peak_diff=row['CR_Early_AM']-row['DB_Early_AM_Peak']
            am_peak_diff=row['CR_AM_Peak']-row['DB_AM_Peak']
            midday_diff=row['CR_Midday']-row['DB_Midday']    
            pm_peak_diff=row['CR_PM_Peak']-row['DB_PM_Peak']
            evening_diff=row['CR_Evening']-row['DB_Evening']
            total_diff=row['CR_Total']-row['DB_Total']
    #         overall_difference=row['CR_Overall_Goal']-row['DB_Total']
            # new_df.loc[index, 'PRE_Early_AM_DIFFERENCE'] = math.ceil(max(0, pre_early_am_peak_diff))
            # new_df.loc[index, 'Early_AM_DIFFERENCE'] = math.ceil(max(0, early_am_peak_diff))
            new_df.loc[index, 'AM_DIFFERENCE'] = math.ceil(max(0, am_peak_diff))
            new_df.loc[index, 'Midday_DIFFERENCE'] = math.ceil(max(0, midday_diff))
            new_df.loc[index, 'PM_DIFFERENCE'] = math.ceil(max(0, pm_peak_diff))
            new_df.loc[index, 'Evening_DIFFERENCE'] = math.ceil(max(0, evening_diff))
            # route_level_df.loc[index, 'Total_DIFFERENCE'] = math.ceil(max(0, total_diff))
            # new_df.loc[index, 'Total_DIFFERENCE'] =math.ceil(max(0, pre_early_am_peak_diff))+math.ceil(max(0, early_am_peak_diff))+math.ceil(max(0, am_peak_diff))+math.ceil(max(0, midday_diff))+math.ceil(max(0, pm_peak_diff))+math.ceil(max(0, evening_diff))
            new_df.loc[index, 'Total_DIFFERENCE'] =math.ceil(max(0, am_peak_diff))+math.ceil(max(0, midday_diff))+math.ceil(max(0, pm_peak_diff))+math.ceil(max(0, evening_diff))

    return new_df

wkend_route_direction_df=create_route_direction_level_df(wkend_overall_df,weekend_df)
# wkday_route_direction_df=create_route_direction_level_df(wkday_overall_df,weekday_df)


In [21]:
wkend_route_direction_df.to_csv('WeekEnd DAY Wise RouteLevel comparison02.csv',index=False)

In [15]:

def create_route_level_df(overall_df,route_df,df):
    # For EMBARK
    # am_values=['AM1','AM2','AM3','MID1','MID2','MID7'] 
    # midday_values=['MID3','MID4','MID5','MID6','PM1','PM2']
    # pm_values=['PM3','PM4','PM5']
    # evening_values=['PM9','PM6','PM7','PM8']
    # for SEATTLE
    # pre_early_am_values=['AM1'] 
    # early_am_values=['AM2'] 
    # am_values=['AM3','AM4','MID1','MID2','MID7'] 
    # midday_values=['MID3','MID4','MID5','MID6','PM1']
    # pm_values=['PM2','PM3','PM4','PM5']
    # evening_values=['PM6','PM7','PM8','PM9']

    # pre_early_am_column=[0]  #0 is for Pre-Early AM header
    # early_am_column=[1]  #1 is for Early AM header
    # am_column=[2] #This is for AM header
    # midday_colum=[3] #this is for MIDDAY header
    # pm_column=[4] #this is for PM header
    # evening_column=[5] #this is for EVENING header

    am_values = ['AM1','AM2','AM3', 'AM4']
    midday_values = [ 'MID1', 'MID2','MID3', 'MID4', 'MID5', 'MID6' ]
    pm_values = ['PM1','PM2', 'PM3']
    evening_values = ['OFF1', 'OFF2', 'OFF3', 'OFF4','OFF5']
    am_column=[1] #This is for AM header
    midday_colum=[2] #this is for MIDDAY header
    pm_column=[3] #this is for PM header
    evening_column=[4] #this is for EVENING header

    def convert_string_to_integer(x):
        try:
            return float(x)
        except (ValueError, TypeError):
            return 0

    # Creating new dataframe for specifically AM, PM, MIDDAY, Evenving data and added values from Compeletion Report
    new_df=pd.DataFrame()
    new_df['ROUTE_SURVEYEDCode']=overall_df['LS_NAME_CODE']
    new_df['Day'] = overall_df['DAY']

    # new_df['CR_PRE_Early_AM']=overall_df[pre_early_am_column[0]].apply(math.ceil)
    # new_df['CR_Early_AM']=overall_df[early_am_column[0]].apply(math.ceil)
    new_df['CR_AM_Peak']=overall_df[am_column[0]].apply(math.ceil)
    new_df['CR_Midday']=overall_df[midday_colum[0]].apply(math.ceil)
    new_df['CR_PM_Peak']=overall_df[pm_column[0]].apply(math.ceil)
    new_df['CR_Evening']=overall_df[evening_column[0]].apply(math.ceil)
    print("new_df_columns",new_df.columns)
    # new_df[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']]=new_df[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']].applymap(convert_string_to_integer)
    new_df[['CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']]=new_df[['CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']].applymap(convert_string_to_integer)
    #  new_df[['CR_EARLY_AM','CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']]=new_df[['CR_EARLY_AM','CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']].applymap(convert_string_to_integer)
    # new_df['Overall Goal']=cr_df[overall_goal_column[0]]
    new_df.fillna(0,inplace=True)
    # adding values for AM, PM, MIDDAY and Evening from Database file to new Dataframe
    for index, row in new_df.iterrows():
        route_code = row['ROUTE_SURVEYEDCode']

        # Define a function to get the counts and IDs
        def get_counts_and_ids(time_values):
            # Just for SALEM
            # subset_df = df[(df['ROUTE_SURVEYEDCode_Splited'] == route_code) & (df[time_column[0]].isin(time_values))]
            subset_df = df[(df['ROUTE_SURVEYEDCode'] == route_code) & (df[time_column[0]].isin(time_values))]
            subset_df=subset_df.drop_duplicates(subset='id')
            count = subset_df.shape[0]
            ids = subset_df['id'].values
            return count, ids
        
        # Calculate counts and IDs for each time slot
        # pre_early_am_value, pre_early_am_value_ids = get_counts_and_ids(pre_early_am_values)
        # early_am_value, early_am_value_ids = get_counts_and_ids(early_am_values)
        am_value, am_value_ids = get_counts_and_ids(am_values)
        midday_value, midday_value_ids = get_counts_and_ids(midday_values)
        pm_value, pm_value_ids = get_counts_and_ids(pm_values)
        evening_value, evening_value_ids = get_counts_and_ids(evening_values)
        
        # Assign values to new_df
        # new_df.loc[index, 'CR_Total'] = row['CR_EARLY_AM'] + row['CR_AM_Peak'] + row['CR_Midday'] + row['CR_PM_Peak'] + row['CR_Evening']
        # new_df.loc[index, 'CR_Total'] = row['CR_PRE_Early_AM']+row['CR_Early_AM']+row['CR_AM_Peak'] + row['CR_Midday'] + row['CR_PM_Peak'] + row['CR_Evening']
        new_df.loc[index, 'CR_Total'] = row['CR_AM_Peak'] + row['CR_Midday'] + row['CR_PM_Peak'] + row['CR_Evening']
        new_df.loc[index, 'CR_AM_Peak'] =row['CR_AM_Peak']
        # new_df.loc[index, 'CR_AM_Peak'] =row['CR_PRE_EARLY_AM']+row['CR_EARLY_AM']+ row['CR_AM_Peak']
        # new_df.loc[index, 'DB_PRE_Early_AM_Peak'] = pre_early_am_value
        # new_df.loc[index, 'DB_Early_AM_Peak'] = early_am_value
        new_df.loc[index, 'DB_AM_Peak'] = am_value
        new_df.loc[index, 'DB_Midday'] = midday_value
        new_df.loc[index, 'DB_PM_Peak'] = pm_value
        new_df.loc[index, 'DB_Evening'] = evening_value
        # new_df.loc[index, 'DB_Total'] = evening_value + am_value + midday_value + pm_value+pre_early_am_value+early_am_value
        new_df.loc[index, 'DB_Total'] = evening_value + am_value + midday_value + pm_value
        
        # Join the IDs as a comma-separated string
        # new_df.loc[index, 'DB_PRE_Early_AM_IDS'] = ', '.join(map(str, pre_early_am_value_ids))
        # new_df.loc[index, 'DB_Early_AM_IDS'] = ', '.join(map(str, early_am_value_ids))
        new_df.loc[index, 'DB_AM_IDS'] = ', '.join(map(str, am_value_ids))
        new_df.loc[index, 'DB_Midday_IDS'] = ', '.join(map(str, midday_value_ids))
        new_df.loc[index, 'DB_PM_IDS'] = ', '.join(map(str, pm_value_ids))
        new_df.loc[index, 'DB_Evening_IDS'] = ', '.join(map(str, evening_value_ids))

    # new_df.to_csv('Time Base Comparison(Over All).csv',index=False)

    # Route Level Comparison
    # Just for SALEM because in SALEM Code values are already splitted
    # new_df['ROUTE_SURVEYEDCode_Splited']=new_df['ROUTE_SURVEYEDCode']
    new_df['ROUTE_SURVEYEDCode_Splited']=new_df['ROUTE_SURVEYEDCode'].apply(lambda x:('_').join(x.split('_')[:-1]) )

    # creating new dataframe for ROUTE_LEVEL_Comparison
    route_level_df=pd.DataFrame()

    unique_routes=new_df['ROUTE_SURVEYEDCode_Splited'].unique()

    route_level_df['ROUTE_SURVEYEDCode']=unique_routes

    route_df.rename(columns={'ROUTE_TOTAL':'CR_Overall_Goal','SURVEY_ROUTE_CODE':'ROUTE_SURVEYEDCode','LS_NAME_CODE':'ROUTE_SURVEYEDCode'},inplace=True)

    route_df.dropna(subset=['ROUTE_SURVEYEDCode'],inplace=True)
    route_level_df=pd.merge(route_level_df,route_df[['ROUTE_SURVEYEDCode','CR_Overall_Goal']],on='ROUTE_SURVEYEDCode')

    # adding values from database file and compeletion report for Route_Level
    for index , row in route_level_df.iterrows():
        subset_df=new_df[new_df['ROUTE_SURVEYEDCode_Splited']==row['ROUTE_SURVEYEDCode']]
        # sum_per_route_cr = subset_df[['CR_AM_Peak', 'CR_Midday', 'CR_PM_Peak', 'CR_Evening','CR_Total','Overall Goal']].sum()
        


        # sum_per_route_cr = subset_df[['CR_PRE_Early_AM','CR_Early_AM','CR_AM_Peak', 'CR_Midday', 'CR_PM_Peak', 'CR_Evening','CR_Total']].sum()
        sum_per_route_cr = subset_df[['CR_AM_Peak', 'CR_Midday', 'CR_PM_Peak', 'CR_Evening','CR_Total']].sum()
        # sum_per_route_cr = subset_df[['CR_EARLY_AM','CR_AM_Peak', 'CR_Midday', 'CR_PM_Peak', 'CR_Evening','CR_Total']].sum()
        # sum_per_route_db = subset_df[['DB_PRE_Early_AM_Peak','DB_Early_AM_Peak','DB_AM_Peak', 'DB_Midday', 'DB_PM_Peak', 'DB_Evening','DB_Total']].sum()
        sum_per_route_db = subset_df[['DB_AM_Peak', 'DB_Midday', 'DB_PM_Peak', 'DB_Evening','DB_Total']].sum()
        
        # route_level_df.loc[index,'CR_PRE_Early_AM']=sum_per_route_cr['CR_PRE_Early_AM']
        # route_level_df.loc[index,'CR_Early_AM']=sum_per_route_cr['CR_Early_AM']
        route_level_df.loc[index,'CR_AM_Peak']=sum_per_route_cr['CR_AM_Peak']
        route_level_df.loc[index,'CR_Midday']=sum_per_route_cr['CR_Midday']
        route_level_df.loc[index,'CR_PM_Peak']=sum_per_route_cr['CR_PM_Peak']
        route_level_df.loc[index,'CR_Evening']=sum_per_route_cr['CR_Evening']
        route_level_df.loc[index,'CR_Total']=sum_per_route_cr['CR_Total']
        # route_level_df.loc[index,'CR_Overall_Goal']=sum_per_route_cr['Overall Goal']
        
        # route_level_df.loc[index,'DB_PRE_Early_AM_Peak']=sum_per_route_db['DB_PRE_Early_AM_Peak']
        # route_level_df.loc[index,'DB_Early_AM_Peak']=sum_per_route_db['DB_Early_AM_Peak']
        route_level_df.loc[index,'DB_AM_Peak']=sum_per_route_db['DB_AM_Peak']
        route_level_df.loc[index,'DB_Midday']=sum_per_route_db['DB_Midday']
        route_level_df.loc[index,'DB_PM_Peak']=sum_per_route_db['DB_PM_Peak']
        route_level_df.loc[index,'DB_Evening']=sum_per_route_db['DB_Evening']
        route_level_df.loc[index,'DB_Total']=sum_per_route_db['DB_Total']   
        # route_level_df.loc[index,'DB_PRE_Early_AM_IDS']=', '.join(str(value) for value in subset_df['DB_PRE_Early_AM_IDS'].values)    
        # route_level_df.loc[index,'DB_Early_AM_IDS']=', '.join(str(value) for value in subset_df['DB_Early_AM_IDS'].values)    
        route_level_df.loc[index,'DB_AM_IDS']=', '.join(str(value) for value in subset_df['DB_AM_IDS'].values)    
        route_level_df.loc[index,'DB_Midday_IDS']=', '.join(str(value) for value in subset_df['DB_Midday_IDS'].values)    
        route_level_df.loc[index,'DB_PM_IDS']=', '.join(str(value) for value in subset_df['DB_PM_IDS'].values)    
        route_level_df.loc[index,'DB_Evening_IDS']=', '.join(str(value) for value in subset_df['DB_Evening_IDS'].values)

    # route_level_df.to_csv('Route Level Comparison(Value_Check).csv',index=False)
        
    # calculating the difference between values of database and compeletion report for Route_Level
    for index, row in route_level_df.iterrows():
        # pre_early_am_peak_diff=row['CR_PRE_Early_AM']-row['DB_PRE_Early_AM_Peak']
        # early_am_peak_diff=row['CR_Early_AM']-row['DB_Early_AM_Peak']
        am_peak_diff=row['CR_AM_Peak']-row['DB_AM_Peak']
        midday_diff=row['CR_Midday']-row['DB_Midday']    
        pm_peak_diff=row['CR_PM_Peak']-row['DB_PM_Peak']
        evening_diff=row['CR_Evening']-row['DB_Evening']
        total_diff=row['CR_Total']-row['DB_Total']
        overall_difference=row['CR_Overall_Goal']-row['DB_Total']
        # route_level_df.loc[index, 'PRE_Early_AM_DIFFERENCE'] = math.ceil(max(0, pre_early_am_peak_diff))
        # route_level_df.loc[index, 'Early_AM_DIFFERENCE'] = math.ceil(max(0, early_am_peak_diff))
        route_level_df.loc[index, 'AM_DIFFERENCE'] = math.ceil(max(0, am_peak_diff))
        route_level_df.loc[index, 'Midday_DIFFERENCE'] = math.ceil(max(0, midday_diff))
        route_level_df.loc[index, 'PM_DIFFERENCE'] = math.ceil(max(0, pm_peak_diff))
        route_level_df.loc[index, 'Evening_DIFFERENCE'] = math.ceil(max(0, evening_diff))
        # route_level_df.loc[index, 'Total_DIFFERENCE'] = math.ceil(max(0, total_diff))
        # route_level_df.loc[index, 'Total_DIFFERENCE'] =math.ceil(max(0, pre_early_am_peak_diff))+math.ceil(max(0, early_am_peak_diff))+math.ceil(max(0, am_peak_diff))+math.ceil(max(0, midday_diff))+math.ceil(max(0, pm_peak_diff))+math.ceil(max(0, evening_diff))
        route_level_df.loc[index, 'Total_DIFFERENCE'] =math.ceil(max(0, am_peak_diff))+math.ceil(max(0, midday_diff))+math.ceil(max(0, pm_peak_diff))+math.ceil(max(0, evening_diff))
        route_level_df.loc[index, 'Overall_Goal_DIFFERENCE'] = math.ceil(max(0,overall_difference))

    return route_level_df

In [14]:
weekday_df[['Day']]

,Day
0,Wednesday
1,Wednesday
2,Wednesday
3,Wednesday
4,Wednesday
...,...
6289,Thursday
6290,Thursday
6291,Thursday
6292,Thursday


In [17]:

# wkday_route_level =create_route_level_df(wkday_overall_df,wkday_route_df,weekday_df)
wkend_route_level =create_route_level_df(wkend_overall_df,wkend_route_df,weekend_df)

new_df_columns Index(['ROUTE_SURVEYEDCode', 'Day', 'CR_AM_Peak', 'CR_Midday', 'CR_PM_Peak',
       'CR_Evening'],
      dtype='object')


In [18]:
wkend_route_level.to_csv('Tucson Weekend Day RouteDirection.csv',index=False)

In [12]:
def create_wkend_route_level_df(overall_df, route_df, df):
    am_values = ['AM1','AM2','AM3', 'AM4']
    midday_values = ['MID1', 'MID2','MID3', 'MID4', 'MID5', 'MID6']
    pm_values = ['PM1','PM2', 'PM3']
    evening_values = ['OFF1', 'OFF2', 'OFF3', 'OFF4','OFF5']
    am_column=[1]
    midday_colum=[2]
    pm_column=[3]
    evening_column=[4]

    def convert_string_to_integer(x):
        try:
            return float(x)
        except (ValueError, TypeError):
            return 0

    # Create separate dataframes for each day
    new_df = pd.DataFrame()
    new_df['ROUTE_SURVEYEDCode'] = overall_df['LS_NAME_CODE']
    new_df['Day'] = overall_df['DAY']
    new_df['CR_AM_Peak'] = overall_df[am_column[0]].apply(math.ceil)
    new_df['CR_Midday'] = overall_df[midday_colum[0]].apply(math.ceil)
    new_df['CR_PM_Peak'] = overall_df[pm_column[0]].apply(math.ceil)
    new_df['CR_Evening'] = overall_df[evening_column[0]].apply(math.ceil)

    new_df[['CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']] = new_df[['CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']].applymap(convert_string_to_integer)
    new_df.fillna(0, inplace=True)
#     new_df.to_csv("First Draft WeekEND.csv",index=False)
    for index, row in new_df.iterrows():
        route_code = row['ROUTE_SURVEYEDCode']
        day = row['Day']
        def get_counts_and_ids(time_values):
            # Filter by both route code AND day
            subset_df = df[(df['ROUTE_SURVEYEDCode'] == route_code) & 
                         (df[time_column[0]].isin(time_values)) & 
                         (df['Day'].str.lower() == str(day).lower())]
            print(route_code,day,subset_df.shape)
            subset_df = subset_df.drop_duplicates(subset='id')
            count = subset_df.shape[0]
            ids = subset_df['id'].values
            return count, ids
        am_value, am_value_ids = get_counts_and_ids(am_values)
        midday_value, midday_value_ids = get_counts_and_ids(midday_values)
        pm_value, pm_value_ids = get_counts_and_ids(pm_values)
        evening_value, evening_value_ids = get_counts_and_ids(evening_values)

        new_df.loc[index, 'CR_Total'] = row['CR_AM_Peak'] + row['CR_Midday'] + row['CR_PM_Peak'] + row['CR_Evening']
        new_df.loc[index, 'DB_AM_Peak'] = am_value
        new_df.loc[index, 'DB_Midday'] = midday_value
        new_df.loc[index, 'DB_PM_Peak'] = pm_value
        new_df.loc[index, 'DB_Evening'] = evening_value
        new_df.loc[index, 'DB_Total'] = evening_value + am_value + midday_value + pm_value

    new_df['ROUTE_SURVEYEDCode_Splited'] = new_df['ROUTE_SURVEYEDCode'].apply(lambda x: '_'.join(x.split('_')[:-1]))
    unique_route_days = new_df[['ROUTE_SURVEYEDCode_Splited', 'Day']].drop_duplicates()
    route_level_df = pd.DataFrame()
    # Create unique combinations of route and day
    unique_route_days = new_df[['ROUTE_SURVEYEDCode_Splited', 'Day']].drop_duplicates()
    route_level_df['ROUTE_SURVEYEDCode'] = new_df['ROUTE_SURVEYEDCode_Splited']
    route_level_df['Day'] = new_df['Day']

    route_df.rename(columns={'ROUTE_TOTAL':'CR_Overall_Goal','SURVEY_ROUTE_CODE':'ROUTE_SURVEYEDCode','LS_NAME_CODE':'ROUTE_SURVEYEDCode','DAY':'Day'}, inplace=True)
    route_df.dropna(subset=['ROUTE_SURVEYEDCode'], inplace=True)
    # route_level_df = pd.merge(route_level_df, route_df[['ROUTE_SURVEYEDCode','CR_Overall_Goal']], on='ROUTE_SURVEYEDCode')
    route_level_df.drop_duplicates(subset=['ROUTE_SURVEYEDCode','Day'],inplace=True)
    for _,row in route_level_df.iterrows():
        print(row['ROUTE_SURVEYEDCode'],row['Day'])
        subset_df = new_df[(new_df['ROUTE_SURVEYEDCode_Splited'] == row['ROUTE_SURVEYEDCode']) & 
                         (new_df['Day'] == row['Day'])]

        sum_per_route_cr = subset_df[['CR_AM_Peak', 'CR_Midday', 'CR_PM_Peak', 'CR_Evening','CR_Total']].sum()
        sum_per_route_db = subset_df[['DB_AM_Peak', 'DB_Midday', 'DB_PM_Peak', 'DB_Evening','DB_Total']].sum()
        print(sum_per_route_cr['CR_AM_Peak'])
        route_level_df.loc[row.name,'CR_AM_Peak'] = sum_per_route_cr['CR_AM_Peak']
        route_level_df.loc[row.name,'CR_Midday'] = sum_per_route_cr['CR_Midday']
        route_level_df.loc[row.name,'CR_PM_Peak'] = sum_per_route_cr['CR_PM_Peak']
        route_level_df.loc[row.name,'CR_Evening'] = sum_per_route_cr['CR_Evening']
        route_level_df.loc[row.name,'CR_Total'] = sum_per_route_cr['CR_Total']

        route_level_df.loc[row.name,'DB_AM_Peak'] = sum_per_route_db['DB_AM_Peak']
        route_level_df.loc[row.name,'DB_Midday'] = sum_per_route_db['DB_Midday']
        route_level_df.loc[row.name,'DB_PM_Peak'] = sum_per_route_db['DB_PM_Peak']
        route_level_df.loc[row.name,'DB_Evening'] = sum_per_route_db['DB_Evening']
        route_level_df.loc[row.name,'DB_Total'] = sum_per_route_db['DB_Total']
    route_level_df = pd.merge(route_level_df, route_df[['ROUTE_SURVEYEDCode', 'Day', 'CR_Overall_Goal']], 
                             on=['ROUTE_SURVEYEDCode', 'Day'])

    for _, row in route_level_df.iterrows():
        am_peak_diff = row['CR_AM_Peak'] - row['DB_AM_Peak']
        midday_diff = row['CR_Midday'] - row['DB_Midday']
        pm_peak_diff = row['CR_PM_Peak'] - row['DB_PM_Peak']
        evening_diff = row['CR_Evening'] - row['DB_Evening']
        total_diff = row['CR_Total'] - row['DB_Total']
        overall_difference = row['CR_Overall_Goal'] - row['DB_Total']

        route_level_df.loc[row.name, 'AM_DIFFERENCE'] = math.ceil(max(0, am_peak_diff))
        route_level_df.loc[row.name, 'Midday_DIFFERENCE'] = math.ceil(max(0, midday_diff))
        route_level_df.loc[row.name, 'PM_DIFFERENCE'] = math.ceil(max(0, pm_peak_diff))
        route_level_df.loc[row.name, 'Evening_DIFFERENCE'] = math.ceil(max(0, evening_diff))
        route_level_df.loc[row.name, 'Total_DIFFERENCE'] = math.ceil(max(0, am_peak_diff)) + math.ceil(max(0, midday_diff)) + math.ceil(max(0, pm_peak_diff)) + math.ceil(max(0, evening_diff))
        route_level_df.loc[row.name, 'Overall_Goal_DIFFERENCE'] = math.ceil(max(0, overall_difference))
    return route_level_df

In [13]:
wkend_route_level =create_wkend_route_level_df(wkend_overall_df,wkend_route_df,weekend_df)

SUN_1_1_00 SATURDAY (0, 783)
SUN_1_1_00 SATURDAY (1, 783)
SUN_1_1_00 SATURDAY (0, 783)
SUN_1_1_00 SATURDAY (0, 783)
SUN_1_1_01 SATURDAY (0, 783)
SUN_1_1_01 SATURDAY (0, 783)
SUN_1_1_01 SATURDAY (0, 783)
SUN_1_1_01 SATURDAY (0, 783)
SUN_1_2_00 SATURDAY (0, 783)
SUN_1_2_00 SATURDAY (9, 783)
SUN_1_2_00 SATURDAY (0, 783)
SUN_1_2_00 SATURDAY (0, 783)
SUN_1_2_01 SATURDAY (0, 783)
SUN_1_2_01 SATURDAY (1, 783)
SUN_1_2_01 SATURDAY (0, 783)
SUN_1_2_01 SATURDAY (0, 783)
SUN_1_3_01 SATURDAY (0, 783)
SUN_1_3_01 SATURDAY (0, 783)
SUN_1_3_01 SATURDAY (0, 783)
SUN_1_3_01 SATURDAY (0, 783)
SUN_1_3_00 SATURDAY (0, 783)
SUN_1_3_00 SATURDAY (0, 783)
SUN_1_3_00 SATURDAY (0, 783)
SUN_1_3_00 SATURDAY (0, 783)
SUN_1_4_01 SATURDAY (4, 783)
SUN_1_4_01 SATURDAY (14, 783)
SUN_1_4_01 SATURDAY (0, 783)
SUN_1_4_01 SATURDAY (0, 783)
SUN_1_4_00 SATURDAY (0, 783)
SUN_1_4_00 SATURDAY (11, 783)
SUN_1_4_00 SATURDAY (5, 783)
SUN_1_4_00 SATURDAY (0, 783)
SUN_1_5_01 SATURDAY (3, 783)
SUN_1_5_01 SATURDAY (2, 783)
SUN_1_5_01 S

SUN_1_24_01 SUNDAY (7, 783)
SUN_1_24_01 SUNDAY (1, 783)
SUN_1_24_01 SUNDAY (0, 783)
SUN_1_25_00 SUNDAY (0, 783)
SUN_1_25_00 SUNDAY (0, 783)
SUN_1_25_00 SUNDAY (0, 783)
SUN_1_25_00 SUNDAY (0, 783)
SUN_1_25_01 SUNDAY (0, 783)
SUN_1_25_01 SUNDAY (0, 783)
SUN_1_25_01 SUNDAY (0, 783)
SUN_1_25_01 SUNDAY (0, 783)
SUN_1_26_01 SUNDAY (0, 783)
SUN_1_26_01 SUNDAY (2, 783)
SUN_1_26_01 SUNDAY (2, 783)
SUN_1_26_01 SUNDAY (0, 783)
SUN_1_26_00 SUNDAY (0, 783)
SUN_1_26_00 SUNDAY (2, 783)
SUN_1_26_00 SUNDAY (2, 783)
SUN_1_26_00 SUNDAY (0, 783)
SUN_1_27_00 SUNDAY (0, 783)
SUN_1_27_00 SUNDAY (0, 783)
SUN_1_27_00 SUNDAY (0, 783)
SUN_1_27_00 SUNDAY (0, 783)
SUN_1_27_01 SUNDAY (0, 783)
SUN_1_27_01 SUNDAY (6, 783)
SUN_1_27_01 SUNDAY (0, 783)
SUN_1_27_01 SUNDAY (0, 783)
SUN_1_29_01 SUNDAY (0, 783)
SUN_1_29_01 SUNDAY (2, 783)
SUN_1_29_01 SUNDAY (0, 783)
SUN_1_29_01 SUNDAY (0, 783)
SUN_1_29_00 SUNDAY (0, 783)
SUN_1_29_00 SUNDAY (3, 783)
SUN_1_29_00 SUNDAY (5, 783)
SUN_1_29_00 SUNDAY (0, 783)
SUN_1_34_00 SUNDAY (

In [14]:
wkend_route_level.to_csv('WeekEnd DAY Wise RouteLevel comparison02.csv',index=False)

In [29]:
am_values = ['AM1','AM2','AM3', 'AM4']
midday_values = ['MID1', 'MID2','MID3', 'MID4', 'MID5', 'MID6']
pm_values = ['PM1','PM2', 'PM3']
evening_values = ['OFF1', 'OFF2', 'OFF3', 'OFF4','OFF5']
am_column=[1]
midday_colum=[2]
pm_column=[3]
evening_column=[4]

def convert_string_to_integer(x):
    try:
        return float(x)
    except (ValueError, TypeError):
        return 0

# Create separate dataframes for each day
new_df = pd.DataFrame()
new_df['ROUTE_SURVEYEDCode'] = wkend_overall_df['LS_NAME_CODE']
new_df['Day'] = wkend_overall_df['DAY']
new_df['CR_AM_Peak'] = wkend_overall_df[am_column[0]].apply(math.ceil)
new_df['CR_Midday'] = wkend_overall_df[midday_colum[0]].apply(math.ceil)
new_df['CR_PM_Peak'] = wkend_overall_df[pm_column[0]].apply(math.ceil)
new_df['CR_Evening'] = wkend_overall_df[evening_column[0]].apply(math.ceil)

new_df[['CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']] = new_df[['CR_AM_Peak','CR_Midday','CR_PM_Peak','CR_Evening']].applymap(convert_string_to_integer)
new_df.fillna(0, inplace=True)
new_df.to_csv("First Draft WeekEND.csv",index=False)


In [51]:
for index, row in new_df.iterrows():
    route_code = row['ROUTE_SURVEYEDCode']
    day = row['Day']
    print()
    def get_counts_and_ids(time_values):
        # Filter by both route code AND day
        subset_df = weekend_df[(df['ROUTE_SURVEYEDCode'] == route_code) & 
                     (weekend_df[time_column[0]].isin(time_values)) & 
                     (weekend_df['Day'].str.lower() == str(day).lower())]
        print(route_code,day,subset_df.shape)
        subset_df = subset_df.drop_duplicates(subset='id')
        count = subset_df.shape[0]
        ids = subset_df['id'].values
        return count, ids
    am_value, am_value_ids = get_counts_and_ids(am_values)
    midday_value, midday_value_ids = get_counts_and_ids(midday_values)
    pm_value, pm_value_ids = get_counts_and_ids(pm_values)
    evening_value, evening_value_ids = get_counts_and_ids(evening_values)
    
    new_df.loc[index, 'CR_Total'] = row['CR_AM_Peak'] + row['CR_Midday'] + row['CR_PM_Peak'] + row['CR_Evening']
    new_df.loc[index, 'DB_AM_Peak'] = am_value
    new_df.loc[index, 'DB_Midday'] = midday_value
    new_df.loc[index, 'DB_PM_Peak'] = pm_value
    new_df.loc[index, 'DB_Evening'] = evening_value
    new_df.loc[index, 'DB_Total'] = evening_value + am_value + midday_value + pm_value
    
new_df.to_csv("First Draft WeekEND.csv",index=False)


SUN_1_1_00 SATURDAY (0, 783)
SUN_1_1_00 SATURDAY (1, 783)
SUN_1_1_00 SATURDAY (0, 783)
SUN_1_1_00 SATURDAY (0, 783)

SUN_1_1_01 SATURDAY (0, 783)
SUN_1_1_01 SATURDAY (0, 783)
SUN_1_1_01 SATURDAY (0, 783)
SUN_1_1_01 SATURDAY (0, 783)

SUN_1_2_00 SATURDAY (0, 783)
SUN_1_2_00 SATURDAY (9, 783)
SUN_1_2_00 SATURDAY (0, 783)
SUN_1_2_00 SATURDAY (0, 783)

SUN_1_2_01 SATURDAY (0, 783)
SUN_1_2_01 SATURDAY (1, 783)
SUN_1_2_01 SATURDAY (0, 783)
SUN_1_2_01 SATURDAY (0, 783)

SUN_1_3_01 SATURDAY (0, 783)
SUN_1_3_01 SATURDAY (0, 783)
SUN_1_3_01 SATURDAY (0, 783)
SUN_1_3_01 SATURDAY (0, 783)

SUN_1_3_00 SATURDAY (0, 783)
SUN_1_3_00 SATURDAY (0, 783)
SUN_1_3_00 SATURDAY (0, 783)
SUN_1_3_00 SATURDAY (0, 783)

SUN_1_4_01 SATURDAY (4, 783)
SUN_1_4_01 SATURDAY (14, 783)
SUN_1_4_01 SATURDAY (0, 783)
SUN_1_4_01 SATURDAY (0, 783)

SUN_1_4_00 SATURDAY (0, 783)
SUN_1_4_00 SATURDAY (11, 783)
SUN_1_4_00 SATURDAY (5, 783)
SUN_1_4_00 SATURDAY (0, 783)

SUN_1_5_01 SATURDAY (3, 783)
SUN_1_5_01 SATURDAY (2, 783)
SUN

SUN_1_10_01 SUNDAY (0, 783)
SUN_1_10_01 SUNDAY (0, 783)
SUN_1_10_01 SUNDAY (0, 783)
SUN_1_10_01 SUNDAY (0, 783)

SUN_1_11_00 SUNDAY (0, 783)
SUN_1_11_00 SUNDAY (0, 783)
SUN_1_11_00 SUNDAY (0, 783)
SUN_1_11_00 SUNDAY (0, 783)

SUN_1_11_01 SUNDAY (0, 783)
SUN_1_11_01 SUNDAY (0, 783)
SUN_1_11_01 SUNDAY (0, 783)
SUN_1_11_01 SUNDAY (0, 783)

SUN_1_12_00 SUNDAY (0, 783)
SUN_1_12_00 SUNDAY (0, 783)
SUN_1_12_00 SUNDAY (0, 783)
SUN_1_12_00 SUNDAY (0, 783)

SUN_1_12_01 SUNDAY (0, 783)
SUN_1_12_01 SUNDAY (0, 783)
SUN_1_12_01 SUNDAY (0, 783)
SUN_1_12_01 SUNDAY (0, 783)

SUN_1_15_00 SUNDAY (0, 783)
SUN_1_15_00 SUNDAY (6, 783)
SUN_1_15_00 SUNDAY (0, 783)
SUN_1_15_00 SUNDAY (0, 783)

SUN_1_15_01 SUNDAY (0, 783)
SUN_1_15_01 SUNDAY (4, 783)
SUN_1_15_01 SUNDAY (0, 783)
SUN_1_15_01 SUNDAY (0, 783)

SUN_1_16_00 SUNDAY (0, 783)
SUN_1_16_00 SUNDAY (13, 783)
SUN_1_16_00 SUNDAY (0, 783)
SUN_1_16_00 SUNDAY (0, 783)

SUN_1_16_01 SUNDAY (0, 783)
SUN_1_16_01 SUNDAY (16, 783)
SUN_1_16_01 SUNDAY (0, 783)
SUN_1_16_0

In [53]:
new_df['ROUTE_SURVEYEDCode_Splited'] = new_df['ROUTE_SURVEYEDCode'].apply(lambda x: '_'.join(x.split('_')[:-1]))

In [55]:
new_df[['ROUTE_SURVEYEDCode_Splited','Day']]

,ROUTE_SURVEYEDCode_Splited,Day
0,SUN_1_1,SATURDAY
1,SUN_1_1,SATURDAY
2,SUN_1_2,SATURDAY
3,SUN_1_2,SATURDAY
4,SUN_1_3,SATURDAY
...,...,...
111,SUN_1_50,SUNDAY
112,SUN_1_61,SUNDAY
113,SUN_1_61,SUNDAY
114,SUN_1_700,SUNDAY


In [56]:
unique_route_days = new_df[['ROUTE_SURVEYEDCode_Splited', 'Day']].drop_duplicates()
unique_route_days

,ROUTE_SURVEYEDCode_Splited,Day
0,SUN_1_1,SATURDAY
2,SUN_1_2,SATURDAY
4,SUN_1_3,SATURDAY
6,SUN_1_4,SATURDAY
8,SUN_1_5,SATURDAY
10,SUN_1_6,SATURDAY
12,SUN_1_7,SATURDAY
14,SUN_1_8,SATURDAY
16,SUN_1_9,SATURDAY
18,SUN_1_10,SATURDAY


In [103]:
route_df=copy.deepcopy(wkend_route_df)

In [104]:
route_level_df = pd.DataFrame()
# Create unique combinations of route and day
unique_route_days = new_df[['ROUTE_SURVEYEDCode_Splited', 'Day']].drop_duplicates()
route_level_df['ROUTE_SURVEYEDCode'] = new_df['ROUTE_SURVEYEDCode_Splited']
route_level_df['Day'] = new_df['Day']

route_df.rename(columns={'ROUTE_TOTAL':'CR_Overall_Goal','SURVEY_ROUTE_CODE':'ROUTE_SURVEYEDCode','LS_NAME_CODE':'ROUTE_SURVEYEDCode','DAY':'Day'}, inplace=True)
route_df.dropna(subset=['ROUTE_SURVEYEDCode'], inplace=True)
# route_level_df = pd.merge(route_level_df, route_df[['ROUTE_SURVEYEDCode','CR_Overall_Goal']], on='ROUTE_SURVEYEDCode')


In [105]:
route_df

,Day,SORT,ROUTE_SURVEYEDCode,LS_NAME,CR_Overall_Goal
0,SATURDAY,1,SUN_1_1,1 - Glenn/Swan,15.220
1,SATURDAY,2,SUN_1_2,2 - Pueblo Gardens,10.400
2,SATURDAY,3,SUN_1_3,3 - 6th St/Wilmot,18.340
3,SATURDAY,4,SUN_1_4,4 - Speedway,51.180
4,SATURDAY,5,SUN_1_5,5 - Pima/West Speedway,8.260
5,SATURDAY,6,SUN_1_6,6 - Euclid/N 1st Ave,29.280
6,SATURDAY,7,SUN_1_7,7 - 22nd St,24.680
7,SATURDAY,8,SUN_1_8,8 - Broadway,65.740
8,SATURDAY,9,SUN_1_9,9 - Grant Road,16.900
9,SATURDAY,10,SUN_1_10,10 - Ruthrauff,12.600


In [106]:
route_level_df.drop_duplicates(subset=['ROUTE_SURVEYEDCode','Day'],inplace=True)

In [107]:
for _,row in route_level_df.iterrows():
    print(row['ROUTE_SURVEYEDCode'],row['Day'])
    subset_df = new_df[(new_df['ROUTE_SURVEYEDCode_Splited'] == row['ROUTE_SURVEYEDCode']) & 
                     (new_df['Day'] == row['Day'])]

    sum_per_route_cr = subset_df[['CR_AM_Peak', 'CR_Midday', 'CR_PM_Peak', 'CR_Evening','CR_Total']].sum()
    sum_per_route_db = subset_df[['DB_AM_Peak', 'DB_Midday', 'DB_PM_Peak', 'DB_Evening','DB_Total']].sum()
    print(sum_per_route_cr['CR_AM_Peak'])
    route_level_df.loc[row.name,'CR_AM_Peak'] = sum_per_route_cr['CR_AM_Peak']
    route_level_df.loc[row.name,'CR_Midday'] = sum_per_route_cr['CR_Midday']
    route_level_df.loc[row.name,'CR_PM_Peak'] = sum_per_route_cr['CR_PM_Peak']
    route_level_df.loc[row.name,'CR_Evening'] = sum_per_route_cr['CR_Evening']
    route_level_df.loc[row.name,'CR_Total'] = sum_per_route_cr['CR_Total']

    route_level_df.loc[row.name,'DB_AM_Peak'] = sum_per_route_db['DB_AM_Peak']
    route_level_df.loc[row.name,'DB_Midday'] = sum_per_route_db['DB_Midday']
    route_level_df.loc[row.name,'DB_PM_Peak'] = sum_per_route_db['DB_PM_Peak']
    route_level_df.loc[row.name,'DB_Evening'] = sum_per_route_db['DB_Evening']
    route_level_df.loc[row.name,'DB_Total'] = sum_per_route_db['DB_Total']
route_level_df = pd.merge(route_level_df, route_df[['ROUTE_SURVEYEDCode', 'Day', 'CR_Overall_Goal']], 
                         on=['ROUTE_SURVEYEDCode', 'Day'])

for _, row in route_level_df.iterrows():
    am_peak_diff = row['CR_AM_Peak'] - row['DB_AM_Peak']
    midday_diff = row['CR_Midday'] - row['DB_Midday']
    pm_peak_diff = row['CR_PM_Peak'] - row['DB_PM_Peak']
    evening_diff = row['CR_Evening'] - row['DB_Evening']
    total_diff = row['CR_Total'] - row['DB_Total']
    overall_difference = row['CR_Overall_Goal'] - row['DB_Total']

    route_level_df.loc[row.name, 'AM_DIFFERENCE'] = math.ceil(max(0, am_peak_diff))
    route_level_df.loc[row.name, 'Midday_DIFFERENCE'] = math.ceil(max(0, midday_diff))
    route_level_df.loc[row.name, 'PM_DIFFERENCE'] = math.ceil(max(0, pm_peak_diff))
    route_level_df.loc[row.name, 'Evening_DIFFERENCE'] = math.ceil(max(0, evening_diff))
    route_level_df.loc[row.name, 'Total_DIFFERENCE'] = math.ceil(max(0, am_peak_diff)) + math.ceil(max(0, midday_diff)) + math.ceil(max(0, pm_peak_diff)) + math.ceil(max(0, evening_diff))
    route_level_df.loc[row.name, 'Overall_Goal_DIFFERENCE'] = math.ceil(max(0, overall_difference))


SUN_1_1 SATURDAY
2.0
SUN_1_2 SATURDAY
2.0
SUN_1_3 SATURDAY
4.0
SUN_1_4 SATURDAY
7.0
SUN_1_5 SATURDAY
2.0
SUN_1_6 SATURDAY
4.0
SUN_1_7 SATURDAY
4.0
SUN_1_8 SATURDAY
10.0
SUN_1_9 SATURDAY
4.0
SUN_1_10 SATURDAY
3.0
SUN_1_11 SATURDAY
8.0
SUN_1_12 SATURDAY
3.0
SUN_1_15 SATURDAY
2.0
SUN_1_16 SATURDAY
7.0
SUN_1_17 SATURDAY
6.0
SUN_1_18 SATURDAY
8.0
SUN_1_19 SATURDAY
2.0
SUN_1_21 SATURDAY
2.0
SUN_1_22 SATURDAY
2.0
SUN_1_23 SATURDAY
3.0
SUN_1_24 SATURDAY
1.0
SUN_1_25 SATURDAY
5.0
SUN_1_26 SATURDAY
2.0
SUN_1_27 SATURDAY
2.0
SUN_1_29 SATURDAY
3.0
SUN_1_34 SATURDAY
4.0
SUN_1_37 SATURDAY
2.0
SUN_1_61 SATURDAY
2.0
SUN_1_700 SATURDAY
51.0
SUN_1_1 SUNDAY
2.0
SUN_1_2 SUNDAY
1.0
SUN_1_3 SUNDAY
3.0
SUN_1_4 SUNDAY
5.0
SUN_1_5 SUNDAY
2.0
SUN_1_6 SUNDAY
2.0
SUN_1_7 SUNDAY
3.0
SUN_1_8 SUNDAY
6.0
SUN_1_9 SUNDAY
2.0
SUN_1_10 SUNDAY
2.0
SUN_1_11 SUNDAY
6.0
SUN_1_12 SUNDAY
2.0
SUN_1_15 SUNDAY
2.0
SUN_1_16 SUNDAY
6.0
SUN_1_17 SUNDAY
4.0
SUN_1_18 SUNDAY
5.0
SUN_1_19 SUNDAY
2.0
SUN_1_21 SUNDAY
2.0
SUN_1_22 SUNDAY
2

In [108]:
route_level_df

,ROUTE_SURVEYEDCode,Day,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,DB_AM_Peak,DB_Midday,DB_PM_Peak,DB_Evening,DB_Total,CR_Overall_Goal,AM_DIFFERENCE,Midday_DIFFERENCE,PM_DIFFERENCE,Evening_DIFFERENCE,Total_DIFFERENCE,Overall_Goal_DIFFERENCE
0,SUN_1_1,SATURDAY,2.0,7.0,5.0,4.0,18.0,0.0,1.0,0.0,0.0,1.0,15.220,2.0,6.0,5.0,4.0,17.0,15.0
1,SUN_1_2,SATURDAY,2.0,6.0,4.0,2.0,14.0,0.0,10.0,0.0,0.0,10.0,10.400,2.0,0.0,4.0,2.0,8.0,1.0
2,SUN_1_3,SATURDAY,4.0,9.0,5.0,4.0,22.0,0.0,0.0,0.0,0.0,0.0,18.340,4.0,9.0,5.0,4.0,22.0,19.0
3,SUN_1_4,SATURDAY,7.0,22.0,12.0,13.0,54.0,4.0,25.0,5.0,0.0,34.0,51.180,3.0,0.0,7.0,13.0,23.0,18.0
4,SUN_1_5,SATURDAY,2.0,5.0,4.0,2.0,13.0,7.0,2.0,1.0,0.0,10.0,8.260,0.0,3.0,3.0,2.0,8.0,0.0
5,SUN_1_6,SATURDAY,4.0,14.0,8.0,7.0,33.0,0.0,0.0,0.0,0.0,0.0,29.280,4.0,14.0,8.0,7.0,33.0,30.0
6,SUN_1_7,SATURDAY,4.0,12.0,8.0,5.0,29.0,0.0,0.0,0.0,0.0,0.0,24.680,4.0,12.0,8.0,5.0,29.0,25.0
7,SUN_1_8,SATURDAY,10.0,32.0,18.0,10.0,70.0,6.0,8.0,1.0,0.0,15.0,65.740,4.0,24.0,17.0,10.0,55.0,51.0
8,SUN_1_9,SATURDAY,4.0,8.0,6.0,4.0,22.0,0.0,12.0,6.0,0.0,18.0,16.900,4.0,0.0,0.0,4.0,8.0,0.0
9,SUN_1_10,SATURDAY,3.0,6.0,4.0,3.0,16.0,0.0,0.0,0.0,0.0,0.0,12.600,3.0,6.0,4.0,3.0,16.0,13.0


In [38]:
subset_df = weekend_df[(weekend_df['ROUTE_SURVEYEDCode'] == 'SUN_1_1_01') & 
             (weekend_df[time_column[0]].isin(am_values)) & 
             (weekend_df['Day'].str.lower() == 'Saturday'.lower())]
subset_df

,id,Completed,Last_page,Start_language,Date_started,Date_last_action,IP_address,Referring_URL,TIME_ADJUST,INTERV_INIT,...,STOP_OFF_CLINTID_NEW,STOP_OFF_LAT_NEW,STOP_OFF_LONG_NEW,ROUTE_SURVEYEDCode,ROUTE_SURVEYED,STOP_ON_ADDRESS_NEW,ROUTE_SURVEYEDCode_Splited,Date,Day,LAST_SURVEY_DATE
